# Task A — Wave 2: Architecture comparison

**Tu patrzysz na wyniki nowych encoderów** (nie na BAZA / E1–E4).

| Etap | Folder |
|------|--------|
| Screening (3 modele × 3 seedy) | `outputs/wave2_architecture_YYYY-MM-DD/` |
| Depth L1–L3 | `outputs/wave2_architecture_YYYY-MM-DD_DEPTH/` |
| Heads | `…_HEADS/` |
| Final 5 seeds | `…_FINAL/` |

Wave 1 (stare modele + ablacje) → `outputs/wave1_baselines_ablations/`  
albo notebooki: `TaskA_BAZA_comparison.ipynb`, `TaskA_FEATURE_ABLATION_comparison.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs'

# Najnowszy folder screeningu wave2
screen_dirs = sorted(OUT.glob('wave2_architecture_????-??-??'))
screen_dirs = [d for d in screen_dirs if d.is_dir() and not d.name.endswith(('_DEPTH','_HEADS','_FINAL'))]
SCREEN = screen_dirs[-1] if screen_dirs else None
print('SCREENING folder:', SCREEN)

DEPTH = sorted(OUT.glob('wave2_architecture_*_DEPTH'))
DEPTH = DEPTH[-1] if DEPTH else None
print('DEPTH folder:', DEPTH)

## 1. Screening — summary (już gotowe)

In [ ]:
if SCREEN is None:
    raise FileNotFoundError('Brak folderu wave2_architecture_YYYY-MM-DD')

summary = next(SCREEN.glob('*ARCH_SCREENING_summary_by_model.csv'))
all_runs = next(SCREEN.glob('*ARCH_SCREENING_all_models.csv'))

s = pd.read_csv(summary)
a = pd.read_csv(all_runs)
display(s)
display(a[['model','training_seed','valid_auprc','test_auprc','valid_brier','parameter_count_trainable']].sort_values(['model','training_seed']))

## 2. Depth (L1–L3)

## 3. Heads + Final

In [ ]:
if DEPTH is None:
    print('DEPTH folder jeszcze nie istnieje / brak runów.')
else:
    files = list(DEPTH.glob('*ARCH_DEPTH_summary.csv'))
    if not files:
        print('DEPTH w toku — brak summary CSV. Log:')
        log = list(DEPTH.glob('*VISIBLE.log')) or list(DEPTH.glob('*run.log'))
        print(log[0] if log else DEPTH)
    else:
        d = pd.read_csv(files[0])
        display(d.sort_values(['model','num_layers']))

In [ ]:
HEADS = sorted(OUT.glob('wave2_architecture_*_HEADS'))
HEADS = HEADS[-1] if HEADS else None
FINAL = sorted(OUT.glob('wave2_architecture_*_FINAL'))
FINAL = FINAL[-1] if FINAL else None
print('HEADS:', HEADS)
print('FINAL:', FINAL)

if HEADS:
    hs = pd.read_csv(next(HEADS.glob('*ARCH_HEADS_summary.csv')))
    display(hs.sort_values('heads'))

if FINAL:
    fs = pd.read_csv(next(FINAL.glob('*ARCH_FINAL_summary.csv')))
    fr = pd.read_csv(next(FINAL.glob('*ARCH_FINAL_all_runs.csv')))
    display(fs)
    piv = fr.pivot(index='training_seed', columns='model', values='valid_auprc')
    delta = piv['hgt'] - piv['hetero_sage_matched']
    print(f"Paired Δ valid AUPRC HGT−SAGE: {delta.mean():+.3f} ± {delta.std():.3f} | wins {(delta>0).sum()}/{len(delta)}")
